# Replikasi Eksperimen Giudici et al. (2020)
## *Network Models to Improve Automated Cryptocurrency Portfolio Management*

Notebook ini membandingkan strategi portofolio berikut:
1. **Equally Weighted (EW)** – Portofolio naif (1/N)
2. **Classical Markowitz (CM)** – Optimasi Mean-Variance tradisional
3. **Glasso Markowitz (GM)** – Markowitz dengan regularisasi Graphical Lasso
4. **Network Markowitz (NW)** – Integrasi RMT dan MST
5. **Graph Diversification** – Diversifikasi berbasis Maximum Independent Set (MIS)

---
## Sel 1 — Persiapan Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.linalg import eigh
from sklearn.covariance import GraphicalLassoCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

---
## Sel 2 — Memuat Data

Memuat data return dan harga dari file Excel. Fokus utama adalah pada dataset return harian untuk 10 aset kripto utama dalam periode **14 September 2017 hingga 17 Oktober 2019**.

In [ ]:
# Load data
excel_file = 'crypto_data_real.xlsx'
df_returns = pd.read_excel(excel_file, sheet_name='Returns', index_col=0)
df_prices  = pd.read_excel(excel_file, sheet_name='Prices',  index_col=0)

crypto_names = df_returns.columns.tolist()
n_assets     = len(crypto_names)

print(f"Data loaded: {df_returns.shape[0]} days, {n_assets} assets")
print(f"Period: {df_returns.index[0]} to {df_returns.index[-1]}")
print(f"Assets: {crypto_names}")

---
## Sel 3 — Statistik Ringkasan (Table 1)

Menghasilkan tabel statistik deskriptif untuk setiap aset kripto: **Mean, Std, Kurtosis, Skewness**.

In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Mean':     df_returns.mean(),
    'Std':      df_returns.std(),
    'Kurtosis': df_returns.kurt(),
    'Skewness': df_returns.skew()
})

print("TABLE 1 | Summary statistics.")
print(summary_stats.round(4))

---
## Sel 4 — Visualisasi Harga Ternormalisasi (Figure 1 & 2)

Mereplikasi *Normalized cryptocurrency price series* dengan mengatur harga awal setiap aset menjadi **100** pada tanggal **7 Januari 2018**.

In [ ]:
# --- Figure 1: BTC, ETH, USDT, BCH, LTC ---
assets_1  = ['BTC', 'ETH', 'USDT', 'BCH', 'LTC']
df_norm_1 = (df_prices.loc['2018-01-07':, assets_1] / df_prices.loc['2018-01-07', assets_1]) * 100
colors_1  = ['black', 'red', 'green', 'blue', 'cyan']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_1.columns):
    plt.plot(df_norm_1.index, df_norm_1[col], label=col, color=colors_1[i])
plt.title('Figure 1 | Normalized Price Series I (BTC, ETH, USDT, BCH, LTC)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 2: XRP, BNB, EOS, XLM, TRX ---
assets_2  = ['XRP', 'BNB', 'EOS', 'XLM', 'TRX']
df_norm_2 = (df_prices.loc['2018-01-07':, assets_2] / df_prices.loc['2018-01-07', assets_2]) * 100
colors_2  = ['magenta', 'gold', 'lightgrey', 'black', 'red']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_2.columns):
    plt.plot(df_norm_2.index, df_norm_2[col], label=col, color=colors_2[i])
plt.title('Figure 2 | Normalized Price Series II (XRP, BNB, EOS, XLM, TRX)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Sel 5 — Visualisasi Minimum Spanning Tree (Figure 3 & 4)

Mereplikasi Figure 3 dari paper acuan yang menunjukkan struktur jaringan aset kripto selama **periode gelembung spekulatif** (Sep 2017 – Jan 2018) dan **periode stabil** (Jun 2019 – Okt 2019).

In [ ]:
# --- Figure 3: MST Speculative Bubble Period ---
df_bubble    = df_returns.loc['2017-09-14':'2018-01-31']
corr_bubble  = df_bubble.corr()
dist_bubble  = np.sqrt(2 * (1 - corr_bubble))

G           = nx.from_pandas_adjacency(dist_bubble)
mst_bubble  = nx.minimum_spanning_tree(G, weight='weight')

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(mst_bubble, seed=42)
nx.draw(mst_bubble, pos, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 3 | MST September 2017 - January 2018', fontsize=12)
plt.tight_layout()
plt.show()

# --- Figure 4: MST Stable Period ---
df_stable   = df_returns.loc['2019-06-01':'2019-10-17']
corr_stable = df_stable.corr()
dist_stable = np.sqrt(2 * (1 - corr_stable))

G2          = nx.from_pandas_adjacency(dist_stable)
mst_stable  = nx.minimum_spanning_tree(G2, weight='weight')

plt.figure(figsize=(10, 8))
pos2 = nx.spring_layout(mst_stable, seed=42)
nx.draw(mst_stable, pos2, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 4 | MST June 2019 - October 2019', fontsize=12)
plt.tight_layout()
plt.show()

---
## Sel 6 — Fungsi Pembantu (Helper Functions)

Implementasi:
- **RMT Filter** – menyaring noise matriks korelasi via Marcenko-Pastur
- **MST Builder** – menghitung matriks jarak antar aset
- **Eigenvector Centrality** – mengukur sentralitas node dalam jaringan
- **Metrik risiko** – VaR, Rachev Ratio, Maximum Drawdown
- **Graph Diversification** – pemilihan aset via Maximum Independent Set

In [ ]:
def apply_rmt_filter(returns_data):
    """Filter noise dari matriks korelasi menggunakan Random Matrix Theory."""
    if isinstance(returns_data, pd.DataFrame):
        data = returns_data.values
    else:
        data = returns_data
    T, N = data.shape
    Q    = T / N
    C    = np.corrcoef(data.T)

    eigenvalues, eigenvectors = eigh(C)
    eigenvalues  = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]

    # Marcenko-Pastur upper bound
    lambda_plus      = 1 + (1/Q) + 2*np.sqrt(1/Q)
    significant_mask = eigenvalues > lambda_plus
    Lambda_filtered  = np.diag(np.where(significant_mask, eigenvalues, 0))
    C_filtered       = eigenvectors @ Lambda_filtered @ eigenvectors.T
    return C_filtered


def build_mst(correlation_matrix):
    """Hitung matriks jarak dari matriks korelasi."""
    distance_matrix = np.sqrt(2 - 2*correlation_matrix)
    np.fill_diagonal(distance_matrix, 0)
    return distance_matrix


def compute_eigenvector_centrality(distance_matrix):
    """Hitung eigenvector centrality dari matriks jarak."""
    adjacency = 1 / (distance_matrix + 1e-8)
    np.fill_diagonal(adjacency, 0)
    eigenvalues, eigenvectors = eigh(adjacency)
    principal_eigenvector = np.abs(eigenvectors[:, -1])
    centrality = principal_eigenvector / principal_eigenvector.sum()
    return centrality


def calculate_var(returns, confidence=0.95):
    """Value at Risk pada tingkat kepercayaan tertentu."""
    return np.percentile(returns, (1 - confidence) * 100)


def calculate_rachev_ratio(returns, alpha=0.10):
    """Rachev Ratio = CVaR_upper(alpha) / CVaR_lower(alpha)."""
    threshold_upper = np.percentile(returns, (1 - alpha) * 100)
    threshold_lower = np.percentile(returns, alpha * 100)
    cvar_upper = returns[returns >= threshold_upper].mean()
    cvar_lower = abs(returns[returns <= threshold_lower].mean())
    return cvar_upper / cvar_lower if cvar_lower > 0 else 0


def calculate_max_drawdown(cumulative_returns):
    """Maximum Drawdown dari seri return kumulatif."""
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdown    = (cumulative_returns - running_max) / running_max
    return drawdown.min()


def get_assets_graph_diversify(returns_window, corr_threshold=0.4):
    """Pilih aset untuk diversifikasi via Maximum Independent Set (MIS)."""
    corr_mat = returns_window.corr()
    G        = nx.Graph()
    assets   = list(returns_window.mean().sort_values(ascending=False).index)
    G.add_nodes_from(assets)
    for i, a1 in enumerate(assets):
        for a2 in assets[i+1:]:
            if abs(corr_mat.loc[a1, a2]) > corr_threshold:
                G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))


print("Helper functions defined successfully!")

---
## Sel New — Analisis Dinamika MST (Figure 5)

Analisis *rolling window* untuk menghitung **ambang batas MST** (*max link distance*) dan **koefisien residuality** sepanjang waktu.

In [ ]:
def calculate_rolling_mst_metrics(returns_df, window=120):
    """Hitung dinamika MST dengan rolling window."""
    dates         = returns_df.index[window:]
    max_links     = []
    residualities = []

    for i in range(window, len(returns_df)):
        window_data  = returns_df.iloc[i - window:i]
        corr_f       = apply_rmt_filter(window_data)
        mst_weights  = build_mst(corr_f)
        max_links.append(np.max(mst_weights))
        residualities.append(np.sum(mst_weights) / (returns_df.shape[1] - 1))

    return pd.DataFrame(
        {'Max Link': max_links, 'Residuality': residualities},
        index=dates
    )


mst_dyn = calculate_rolling_mst_metrics(df_returns)

# Visualisasi dengan dual axis
fig, ax1 = plt.subplots(figsize=(12, 7))
ax1.plot(mst_dyn.index, mst_dyn['Max Link'], color='black', label='Max Link')
ax1.set_xlabel('Date')
ax1.set_ylabel('Max Link', color='black')
ax1.tick_params(axis='y', labelcolor='black')

ax2 = ax1.twinx()
ax2.plot(mst_dyn.index, mst_dyn['Residuality'], color='red', label='Residuality')
ax2.set_ylabel('Residuality', color='red')
ax2.tick_params(axis='y', labelcolor='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Figure 5 | MST Dynamics: Max Link & Residuality over Time')
plt.tight_layout()
plt.show()

---
## Sel 7 — Implementasi Strategi Portofolio

Setiap strategi diimplementasikan sebagai kelas yang mewarisi `PortfolioStrategy`. Network Markowitz mencakup penalti sentralitas via parameter **γ (gamma)**.

In [ ]:
class PortfolioStrategy:
    def __init__(self, name):
        self.name            = name
        self.weights_history = []
        self.returns_history = []

    def get_weights(self, returns_data):
        raise NotImplementedError


class EquallyWeighted(PortfolioStrategy):
    """Portofolio naif 1/N."""
    def get_weights(self, returns_data):
        n = returns_data.shape[1]
        return np.ones(n) / n


class ClassicalMarkowitz(PortfolioStrategy):
    """Optimasi Mean-Variance tradisional (minimasi varians portofolio)."""
    def get_weights(self, returns_data):
        n_assets    = returns_data.shape[1]
        mu          = returns_data.mean().values
        S           = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP', bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets


class GlassoMarkowitz(PortfolioStrategy):
    """Markowitz dengan matriks presisi yang diestimasi via Graphical Lasso."""
    def get_weights(self, returns_data):
        n_assets = returns_data.shape[1]
        mu       = returns_data.mean().values
        try:
            glasso = GraphicalLassoCV()
            glasso.fit(returns_data.values)
            S = glasso.covariance_
        except Exception:
            S = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP', bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets


class NetworkMarkowitz(PortfolioStrategy):
    """Network Markowitz: RMT + MST + penalti sentralitas eigenvector."""
    def __init__(self, name="Network Markowitz", gamma=0):
        super().__init__(name)
        self.gamma = gamma

    def get_weights(self, returns_data):
        n_assets = returns_data.shape[1]
        mu       = returns_data.mean().values
        sig      = returns_data.std().values
        Cf       = apply_rmt_filter(returns_data)
        dist     = build_mst(Cf)
        cent     = compute_eigenvector_centrality(dist)
        Sf       = np.outer(sig, sig) * Cf
        objective   = lambda w: w @ Sf @ w + self.gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP', bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets


class GraphDiversification(PortfolioStrategy):
    """Strategi diversifikasi berbasis graf menggunakan Maximum Independent Set."""
    def __init__(self, name='Graph Diversification', corr_threshold=0.4):
        super().__init__(name)
        self.corr_threshold = corr_threshold

    def get_weights(self, r):
        n      = r.shape[1]
        assets = r.columns.tolist()
        sel    = get_assets_graph_diversify(r, self.corr_threshold)
        w      = np.zeros(n)
        if sel:
            for a in sel:
                w[assets.index(a)] = 1.0 / len(sel)
        else:
            w = np.ones(n) / n
        return w


print("Portfolio strategy classes defined successfully!")

---
## Sel 8 — Framework Backtesting

Sistem pengujian menggunakan *rolling window* dengan:
- **Window training**: 120 hari
- **Frekuensi rebalancing**: 7 hari
- **Biaya transaksi**: 0.1% (10 basis points)

In [ ]:
def backtest_strategy(strategy, df_returns, window_size=120, rebalance_freq=7, transaction_cost=0.001):
    """Simulasi backtest dengan rolling window dan biaya transaksi."""
    portfolio_returns = []
    dates             = []

    for i in range(window_size, len(df_returns), rebalance_freq):
        train = df_returns.iloc[i - window_size:i]
        w     = strategy.get_weights(train)

        test_end  = min(i + rebalance_freq, len(df_returns))
        test_data = df_returns.iloc[i:test_end]

        for j in range(len(test_data)):
            daily_ret = np.dot(w, test_data.iloc[j].values)
            if j == 0 and len(portfolio_returns) > 0:
                daily_ret -= transaction_cost
            portfolio_returns.append(daily_ret)
            dates.append(test_data.index[j])

    res_df = pd.DataFrame({'date': dates, 'return': portfolio_returns})
    res_df['cumulative_return'] = (1 + res_df['return']).cumprod()

    return {
        'strategy':           strategy.name,
        'returns':            np.array(portfolio_returns),
        'cumulative_returns': res_df['cumulative_return'].values,
        'results_df':         res_df
    }


print("Backtest framework defined successfully!")

---
## Sel 9 — Eksekusi Backtesting

Menjalankan backtest untuk semua strategi, termasuk berbagai nilai penalti **γ** untuk Network Markowitz.

In [ ]:
# --- Inisialisasi strategi ---
gamma_values = [0.005, 0.025, 0.05, 0.15, 0.7, 1.0]

strategies = [
    EquallyWeighted("EW"),
    ClassicalMarkowitz("CM"),
    GlassoMarkowitz("GM"),
    NetworkMarkowitz("NW (gamma=0)", gamma=0),
] + [
    NetworkMarkowitz(f"NW (gamma={g})", gamma=g) for g in gamma_values
] + [
    GraphDiversification('Graph Divers. (theta=0.4)', corr_threshold=0.4),
    GraphDiversification('Graph Divers. (theta=0.5)', corr_threshold=0.5),
]

# --- Eksekusi backtest ---
results = {}
for strat in strategies:
    print(f"Running backtest: {strat.name} ...")
    results[strat.name] = backtest_strategy(strat, df_returns)

print("\nAll backtests completed!")

---
## Sel 10 — Analisis Performa Periodik (Table 2)

Tabel perbandingan **Cumulative Profit & Loss** yang disampel setiap 4 bulan (Januari, Mei, September).

In [ ]:
target_dates = [
    '2018-01-31', '2018-05-31', '2018-09-30',
    '2019-01-31', '2019-05-31', '2019-09-30'
]

table2_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        idx     = df_res.index.searchsorted(pd.Timestamp(td))
        idx     = min(idx, len(df_res) - 1)
        cum_ret = (df_res['cumulative_return'].iloc[idx] - 1) * 100
        row[td] = round(cum_ret, 2)
    table2_rows.append(row)

table2 = pd.DataFrame(table2_rows).set_index('Strategy')
table2.columns = ['Jan-18', 'May-18', 'Sep-18', 'Jan-19', 'May-19', 'Sep-19']

print("TABLE 2 | Cumulative Profits and Losses (%)")
print(table2.to_string())

---
## Sel 11 — Visualisasi Performa Kumulatif (Figure 6)

Evolusi nilai portofolio dengan **investasi awal 100 USD** untuk periode **7 Januari 2018 – 17 Oktober 2019**.

In [ ]:
plt.figure(figsize=(14, 7))
for name, res in results.items():
    # Kalikan cumulative_returns (basis 1.0) dengan 100 untuk skala USD
    plt.plot(res['results_df']['date'],
             res['cumulative_returns'] * 100,
             label=name)

plt.title('FIGURE 6 | Performances of Different Portfolio Strategies')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Sel 12 — Analisis Risiko Periodik (Table 3 — VaR)

**Value at Risk (VaR)** pada tingkat kepercayaan 95%, dihitung untuk jendela 4 bulan terakhir di setiap titik sampling. Nilai disajikan dalam nilai absolut dikali faktor skala 100.

In [ ]:
table3_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        end_idx   = df_res.index.searchsorted(pd.Timestamp(td))
        end_idx   = min(end_idx, len(df_res) - 1)
        start_idx = max(0, end_idx - 120)  # ~4 bulan
        returns_segment = df_res['return'].iloc[start_idx:end_idx].values
        if len(returns_segment) > 0:
            v = abs(calculate_var(returns_segment, 0.95)) * 100
        else:
            v = np.nan
        row[td] = round(v, 4)
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows).set_index('Strategy')
table3.columns = ['Jan-18', 'May-18', 'Sep-18', 'Jan-19', 'May-19', 'Sep-19']

print("TABLE 3 | Value at Risk (95%, 4-months window, scale x100)")
print(table3.to_string())

---
## Sel 13 — Analisis Risk-Adjusted Return (Table 4 — Sharpe Ratio)

**Sharpe Ratio** = Mean Return / Std Return, dihitung untuk jendela 4 bulan di setiap titik sampling.

In [ ]:
table4_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        end_idx   = df_res.index.searchsorted(pd.Timestamp(td))
        end_idx   = min(end_idx, len(df_res) - 1)
        start_idx = max(0, end_idx - 120)
        returns_segment = df_res['return'].iloc[start_idx:end_idx].values
        if len(returns_segment) > 0 and np.std(returns_segment) > 0:
            sr = np.mean(returns_segment) / np.std(returns_segment)
        else:
            sr = np.nan
        row[td] = round(sr, 4)
    table4_rows.append(row)

table4 = pd.DataFrame(table4_rows).set_index('Strategy')
table4.columns = ['Jan-18', 'May-18', 'Sep-18', 'Jan-19', 'May-19', 'Sep-19']

print("TABLE 4 | Sharpe Ratio (4-months window)")
print(table4.to_string())

---
## Sel 14 — Analisis Tail-Risk (Table 5 — Rachev Ratio)

**Rachev Ratio (RR)** = CVaR_upper(10%) / CVaR_lower(10%). Memberikan wawasan tentang asimetri distribusi imbal hasil pada ekor (*tails*), yang umum ditemukan di pasar kripto.

In [ ]:
table5_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        end_idx   = df_res.index.searchsorted(pd.Timestamp(td))
        end_idx   = min(end_idx, len(df_res) - 1)
        start_idx = max(0, end_idx - 120)
        returns_segment = df_res['return'].iloc[start_idx:end_idx].values
        if len(returns_segment) > 0:
            rr = calculate_rachev_ratio(returns_segment, alpha=0.10)
        else:
            rr = np.nan
        row[td] = round(rr, 4)
    table5_rows.append(row)

table5 = pd.DataFrame(table5_rows).set_index('Strategy')
table5.columns = ['Jan-18', 'May-18', 'Sep-18', 'Jan-19', 'May-19', 'Sep-19']

print("TABLE 5 | Rachev Ratio (4-months window, alpha=10%)")
print(table5.to_string())